# DS102 — v3: SVM, NB-SVM và CatBoost

Chạy bằng **CPU**, không sử dụng pretrained model hoặc dữ liệu ngoài. Tham số đã được chốt bằng 3-fold cross-validation trên train. Notebook huấn luyện các mô hình trên toàn bộ train, đánh giá dev và tạo `submission_v3.csv`; không chạy lại quá trình tìm tham số. Các kết quả CV tham chiếu trong báo cáo là kết quả thí nghiệm đã chạy trước đó trên đúng checksum dữ liệu.

Import notebook → **Add Input** dữ liệu cuộc thi → **Run All**. Không cần upload các file `.py`. Cần numpy, pandas, scipy, scikit-learn và catboost. Bản local đã kiểm tra với scikit-learn 1.7.2 và catboost 1.2.10. Nếu thiếu CatBoost, cài trong một cell `%pip install catboost` trước khi chạy, hoặc dùng môi trường đã có sẵn thư viện.

Mô hình kết hợp 50% điểm chuẩn hóa của v2 và 50% điểm CatBoost. Không dùng ID làm đặc trưng; không dùng nhãn dev hay test để huấn luyện. Dev được dùng đánh giá trong quá trình phát triển nhiều phiên bản, nên điểm dev không thay thế điểm leaderboard. V2 đạt 0,75946 trên Kaggle theo người dùng; điểm v3 chưa biết cho đến khi nộp.


## 1. Đọc dữ liệu và đặc trưng

In [ ]:
"""CPU baseline cho DS102: TF-IDF + Linear SVM, chọn theo F1-macro.

Chỉ dùng comment và nhãn train. Dev được đánh giá sau khi chọn tham số
bằng cross-validation trên train; không gộp dev vào train.
"""

import argparse
import html
import json
import platform
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


SEED = 102
LABELS = [0, 1, 2, 3]
FILES = {name: f"ds102_{name}.csv" for name in ["train", "dev", "test", "samplesubmission"]}


def normalize_text(value):
    """Giữ dấu, emoji, dấu câu và ký tự lặp vì có thể giúp nhận diện spam."""
    text = html.unescape(unicodedata.normalize("NFC", str(value)))
    text = text.replace("\\r\\n", "\n").replace("\\n", "\n").replace("\\r", "\n")
    return re.sub(r"\s+", " ", text).strip().lower()


class ReviewStats(BaseEstimator, TransformerMixin):
    """Đặc trưng độ dài, lặp ký tự và tỷ lệ ký tự; không có từ khóa nhãn."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        rows = []
        for text in X:
            chars = len(text)
            words = text.split()
            nwords = len(words)
            denom = max(chars, 1)
            word_denom = max(nwords, 1)
            letters = [c for c in text if c.isalpha()]
            longest_run = max((len(m.group()) for m in re.finditer(r"(.)\1*", text)), default=0)
            rows.append([
                np.log1p(chars), np.log1p(nwords), chars / word_denom,
                len(set(words)) / word_denom, len(set(text)) / denom,
                longest_run / denom, np.log1p(longest_run),
                sum(c.isdigit() for c in text) / denom,
                len(letters) / denom,
                sum(not c.isalnum() and not c.isspace() for c in text) / denom,
                sum(ord(c) > 0xFFFF for c in text) / denom,
                text.count("!") / denom, text.count("?") / denom,
                float(bool(re.search(r"https?://|www\.", text))),
            ])
        return sparse.csr_matrix(np.asarray(rows, dtype=np.float32))


def make_features():
    return FeatureUnion([
        ("word", TfidfVectorizer(
            ngram_range=(1, 2), min_df=2, max_df=1.0, max_features=90000,
            sublinear_tf=True, lowercase=False, dtype=np.float32,
            token_pattern=r"(?u)\b\w+\b",
        )),
        ("char", TfidfVectorizer(
            analyzer="char", ngram_range=(2, 5), min_df=3, max_features=160000,
            sublinear_tf=True, lowercase=False, dtype=np.float32,
        )),
        ("stats", Pipeline([
            ("extract", ReviewStats()), ("scale", StandardScaler(with_mean=False)),
        ])),
    ], transformer_weights={"word": 1.0, "char": 1.0, "stats": 0.1})


def make_model(config, y):
    weight = config["weight"]
    if weight == "sqrt_balanced":
        counts = pd.Series(y).value_counts()
        weight = {int(k): float(np.sqrt(len(y) / (len(counts) * v))) for k, v in counts.items()}
    elif weight == "none":
        weight = None
    return LinearSVC(C=config["C"], class_weight=weight, dual=True,
                     max_iter=10000, tol=1e-4, random_state=SEED)


def find_data_dir(supplied=None):
    if supplied:
        path = Path(supplied).expanduser().resolve()
        if all((path / f).is_file() for f in FILES.values()):
            return path
        raise FileNotFoundError(f"Thiếu một trong 4 file CSV tại {path}")
    candidates = [Path.cwd(), Path.cwd() / "data", Path.cwd() / "dataset"]
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        candidates += sorted({p.parent for p in kaggle.rglob(FILES["train"])})
    matches = [p for p in candidates if all((p / f).is_file() for f in FILES.values())]
    if len(matches) == 1:
        return matches[0].resolve()
    raise FileNotFoundError("Hãy truyền --data-dir đến thư mục chứa đúng 4 CSV; trên Kaggle dùng Add Input trước.")


def load_data(data_dir):
    data = {name: pd.read_csv(data_dir / file, keep_default_na=False) for name, file in FILES.items()}
    for name in ["train", "dev", "test"]:
        required = {"comment", "spam_label"} if name != "test" else {"comment", "id"}
        if not required.issubset(data[name].columns):
            raise ValueError(f"{name}: thiếu cột {required - set(data[name].columns)}")
        if data[name].empty:
            raise ValueError(f"{name} rỗng")
    for name in ["train", "dev"]:
        values = pd.to_numeric(data[name]["spam_label"], errors="raise")
        if not values.isin(LABELS).all():
            raise ValueError(f"{name}: nhãn phải thuộc {LABELS}")
        data[name]["spam_label"] = values.astype(int)
    if set(data["train"]["spam_label"]) != set(LABELS):
        raise ValueError("Train phải có đủ 4 nhãn")
    sample = data["samplesubmission"]
    if list(sample.columns) != ["ID", "spam_label"]:
        raise ValueError("Sample phải có hai cột ID,spam_label")
    test_ids, sample_ids = data["test"]["id"].astype(str), sample["ID"].astype(str)
    if not test_ids.is_unique or not sample_ids.is_unique or set(test_ids) != set(sample_ids):
        raise ValueError("ID trong test và sample phải duy nhất và trùng tập hợp")
    return data


def metrics(y, pred):
    return {
        "macro_f1": float(f1_score(y, pred, labels=LABELS, average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y, pred)),
        "classification_report": classification_report(y, pred, labels=LABELS, output_dict=True, zero_division=0),
        "confusion_matrix": confusion_matrix(y, pred, labels=LABELS).tolist(),
    }



In [ ]:
"""Đặc trưng v2: văn bản review và mức trùng với thông tin sản phẩm."""
import re
from collections import Counter
import numpy as np
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


def prepare(frame):
    frame = frame.copy()
    for col in ['comment', 'product_name', 'description', 'category', 'categories']:
        frame[col] = frame[col].fillna('').astype(str).map(normalize_text)
    return frame


def tokens(text):
    return re.findall(r'(?u)\b\w+\b', text)


def ngrams(words, n):
    return set(zip(*(words[i:] for i in range(n))))


def dense_features(df):
    basic = ReviewStats().transform(df.comment).toarray()
    extra = []
    for row in df.itertuples(index=False):
        text = row.comment
        words = tokens(text)
        wl = [len(w) for w in words] or [0]
        wc = Counter(words)
        n = max(len(words), 1)
        feats = [np.log1p(max(wl)), np.mean(wl), np.std(wl),
                 sum(len(w) > 12 for w in words) / n,
                 max(wc.values(), default=0) / n,
                 sum(v * (v - 1) for v in wc.values()) / max(n * (n - 1), 1),
                 text.count('.') / max(len(text), 1),
                 len(re.findall(r'(.)\1{3,}', text)) / n,
                 np.log1p(len(text.split())),
                 float(row.rating) / 5,
                 np.log1p(max(float(row.num_sold), 0)),
                 np.log1p(max(float(row.num_review), 0))]
        for context in [row.product_name, row.description]:
            context_words = tokens(context)
            for size in [1, 2, 3, 4]:
                a, b = ngrams(words, size), ngrams(context_words, size)
                overlap = len(a & b)
                ratio = overlap / max(len(a), 1)
                feats += [ratio, ratio ** 2, ratio * np.log1p(len(words)), np.log1p(overlap)]
            feats += [float(len(text) > 10 and text in context), np.log1p(len(context))]
        extra.append(feats)
    return np.hstack([basic, np.asarray(extra, dtype=np.float32)])


class FeaturesV2:
    def __init__(self):
        self.word = TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_features=140000,
                                   sublinear_tf=True, lowercase=False, dtype=np.float32,
                                   token_pattern=r'(?u)\b\w+\b')
        self.char = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), min_df=3,
                                   max_features=180000, sublinear_tf=True,
                                   lowercase=False, dtype=np.float32)
        self.product = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=20000,
                                      sublinear_tf=True, lowercase=False, dtype=np.float32)
        self.category = OneHotEncoder(handle_unknown='ignore', dtype=np.float32)
        self.scale = StandardScaler()

    def fit_transform(self, df):
        text = sparse.hstack([self.word.fit_transform(df.comment), self.char.fit_transform(df.comment)], format='csr')
        meta = sparse.hstack([self.product.fit_transform(df.product_name) * .2,
                             self.category.fit_transform(df[['category', 'rating']].astype(str)) * .2,
                             sparse.csr_matrix(self.scale.fit_transform(dense_features(df))) * .15], format='csr')
        return text, meta

    def transform(self, df):
        text = sparse.hstack([self.word.transform(df.comment), self.char.transform(df.comment)], format='csr')
        meta = sparse.hstack([self.product.transform(df.product_name) * .2,
                             self.category.transform(df[['category', 'rating']].astype(str)) * .2,
                             sparse.csr_matrix(self.scale.transform(dense_features(df))) * .15], format='csr')
        return text, meta


def nb_ratios(X, y):
    """Log-count ratio one-vs-rest, fit chỉ từ phần train của fold."""
    ratios = []
    for c in range(4):
        pos = np.asarray(X[y == c].sum(axis=0)).ravel() + 1
        neg = np.asarray(X[y != c].sum(axis=0)).ravel() + 1
        ratios.append(np.log((pos / pos.sum()) / (neg / neg.sum())))
    return ratios

## 2. SVM và NB-SVM

In [ ]:
"""DS102 v2: đặc trưng quan hệ review/sản phẩm, SVM và calibration F1-macro."""
import argparse
import hashlib
import json
import platform
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from scipy import sparse
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import LinearSVC


# Cấu hình được chốt từ cross-validation trên train, trước đánh giá dev v2.
RECIPE = [
    {'kind': 'svm', 'C': .5, 'weight': 'balanced', 'ensemble_weight': .75},
    {'kind': 'nb', 'C': .5, 'weight': 'balanced', 'ensemble_weight': .25},
]


def fit_predict(fit_df, predict_frames, recipe):
    """Không nhận nhãn của các tập dự đoán."""
    y = fit_df.spam_label.to_numpy()
    features = FeaturesV2()
    Xt, Mt = features.fit_transform(fit_df)
    transformed = [features.transform(df) for df in predict_frames]
    result = [np.zeros((len(df), 4)) for df in predict_frames]
    ratios = nb_ratios(Xt, y) if any(c['kind'] == 'nb' for c in recipe) else None
    for config in recipe:
        if config['kind'] == 'svm':
            model = make_model(config, y).fit(sparse.hstack([Xt, Mt], format='csr'), y)
            if np.max(model.n_iter_) >= model.max_iter:
                raise RuntimeError('SVM chưa hội tụ; tăng max_iter trước khi xuất kết quả.')
            for j, (Xv, Mv) in enumerate(transformed):
                result[j] += config['ensemble_weight'] * model.decision_function(sparse.hstack([Xv, Mv], format='csr'))
        elif config['kind'] == 'nb':
            for cls, r in enumerate(ratios):
                model = LinearSVC(C=config['C'], class_weight='balanced', dual=True,
                                  max_iter=30000, random_state=SEED)
                model.fit(sparse.hstack([Xt.multiply(r), Mt], format='csr'), (y == cls).astype(int))
                if np.max(model.n_iter_) >= model.max_iter:
                    raise RuntimeError('NB-SVM chưa hội tụ; tăng max_iter trước khi xuất kết quả.')
                for j, (Xv, Mv) in enumerate(transformed):
                    result[j][:, cls] += config['ensemble_weight'] * model.decision_function(
                        sparse.hstack([Xv.multiply(r), Mv], format='csr'))
        else:
            raise ValueError(f"Model không được hỗ trợ: {config['kind']}")
    return result



## 3. CatBoost và ghép kết quả

In [ ]:
"""CatBoost text + thông tin review/sản phẩm, không dùng mô hình pretrained."""
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier


def cat_features(df):
    a = dense_features(df)
    x = pd.DataFrame(a, columns=[f'stat_{i}' for i in range(a.shape[1])])
    x['comment'] = [' '.join(tokens(t)) or 'emptyreview' for t in df.comment]
    x['category'] = df.category.astype(str).to_numpy()
    x['rating'] = df.rating.astype(str).to_numpy()
    return x


def make_cat(weight='SqrtBalanced', iterations=500):
    return CatBoostClassifier(
        iterations=iterations, depth=5, learning_rate=.1, l2_leaf_reg=5,
        loss_function='MultiClass', auto_class_weights=weight,
        thread_count=4, random_seed=SEED, allow_writing_files=False,
        text_features=['comment'], cat_features=['category', 'rating'],
        tokenizers=[{'tokenizer_id':'Space', 'separator_type':'ByDelimiter', 'delimiter':' '}],
        dictionaries=[{'dictionary_id':'Word', 'gram_order':'1', 'occurrence_lower_bound':'3', 'max_dictionary_size':'40000'},
                      {'dictionary_id':'Bigram', 'gram_order':'2', 'occurrence_lower_bound':'3', 'max_dictionary_size':'40000'}],
        feature_calcers=['BoW:top_tokens_count=700', 'NaiveBayes', 'BM25'],
        verbose=False,
    )

In [ ]:
"""DS102 v3: trung bình điểm v2 và CatBoost; tham số đã chốt bằng CV train."""
import argparse
import hashlib
import json
import platform
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import catboost
from scipy.special import softmax


CAT_FRACTION = .5
TEMPERATURE_MULTIPLIER = 3.
REFERENCE_TRAIN_SHA = '15202b19b2dc2c6c89afeb209024bb0dc77ddf927de44381a96c24a961d823a5'


def main_v3(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--data-dir', default=None)
    parser.add_argument('--output-dir', default='output_v3')
    args = parser.parse_args(argv)
    data_dir = find_data_dir(args.data_dir)
    data = load_data(data_dir)
    columns = ['comment','product_name','description','category','categories','rating','num_sold','num_review']
    for name in ['train','dev','test']:
        if not set(columns).issubset(data[name].columns):
            raise ValueError(f'{name}: thiếu cột dữ liệu cho v3')
        for col in ['rating','num_sold','num_review']:
            data[name][col] = pd.to_numeric(data[name][col], errors='raise')
            if not np.isfinite(data[name][col]).all() or (data[name][col] < 0).any():
                raise ValueError(f'{name}.{col}: giá trị không hợp lệ')
    train,dev,test = [prepare(data[n]) for n in ['train','dev','test']]
    output = Path(args.output_dir).expanduser().resolve()
    output.mkdir(parents=True,exist_ok=True)
    y = train.spam_label.to_numpy()
    print('Huấn luyện SVM + NB-SVM trên train...',flush=True)
    base_dev,base_test = fit_predict(train,[dev.drop(columns='spam_label'),test],RECIPE)
    print('Huấn luyện CatBoost trên train...',flush=True)
    model = make_cat('SqrtBalanced',iterations=500)
    model.fit(cat_features(train),y,verbose=250)
    if not np.array_equal(model.classes_,np.asarray(LABELS)):
        raise ValueError('Sai thứ tự lớp CatBoost')
    cat_dev = model.predict_proba(cat_features(dev))
    cat_test = model.predict_proba(cat_features(test))
    dev_scores = (1-CAT_FRACTION)*softmax(TEMPERATURE_MULTIPLIER*base_dev,axis=1)+CAT_FRACTION*cat_dev
    test_scores = (1-CAT_FRACTION)*softmax(TEMPERATURE_MULTIPLIER*base_test,axis=1)+CAT_FRACTION*cat_test
    dev_pred,test_pred = dev_scores.argmax(1),test_scores.argmax(1)
    y_dev = dev.spam_label.to_numpy()
    dev_metrics = metrics(y_dev,dev_pred)
    baseline_metrics = metrics(y_dev,base_dev.argmax(1))
    unseen = ~dev.comment.isin(set(train.comment)).to_numpy()
    print(f"Dev v2: {baseline_metrics['macro_f1']:.5f}; dev v3: {dev_metrics['macro_f1']:.5f}",flush=True)
    by_id = pd.Series(test_pred,index=test.id.astype(str))
    submission = data['samplesubmission'].copy()
    submission['spam_label'] = submission.ID.astype(str).map(by_id)
    if submission.spam_label.isna().any() or not submission.spam_label.isin(LABELS).all():
        raise ValueError('Submission có nhãn không hợp lệ')
    submission['spam_label'] = submission.spam_label.astype(int)
    csv_path = output/'submission_v3.csv'
    submission.to_csv(csv_path,index=False)
    saved = pd.read_csv(csv_path)
    pd.testing.assert_frame_equal(saved,submission,check_dtype=False)
    if len(saved)!=len(test) or not saved.ID.equals(data['samplesubmission'].ID):
        raise ValueError('Sai số dòng/thứ tự ID')
    pd.DataFrame({'comment':data['dev'].comment,'true_label':y_dev,'v2_label':base_dev.argmax(1),
                  'v3_label':dev_pred,'comment_seen_in_train':~unseen}).to_csv(output/'dev_predictions_v3.csv',index=False)
    np.savez_compressed(output/'scores_v3.npz',v2_dev=base_dev,v2_test=base_test,
                        cat_dev=cat_dev,cat_test=cat_test,dev_scores=dev_scores,test_scores=test_scores)
    hashes={name:hashlib.sha256((data_dir/f'ds102_{name}.csv').read_bytes()).hexdigest() for name in ['train','dev','test']}
    reference = {'source':'Previously completed 3-fold grouped CV; not rerun by this training script',
                 'v2_oof_macro_f1':.7589770118322317,'v3_oof_macro_f1':.7651232845255038}
    if hashes['train']!=REFERENCE_TRAIN_SHA:
        reference=None
    report={
        'seed':SEED,'python':platform.python_version(),'sklearn':sklearn.__version__,'catboost':catboost.__version__,
        'v2_recipe':RECIPE,'catboost_params':model.get_params(),'cat_fraction':CAT_FRACTION,
        'svm_score_multiplier':TEMPERATURE_MULTIPLIER,'data_sha256':hashes,
        'fit_data':'train only; dev/test not used to fit models or features; no pretrained models',
        'reference_cv':reference,'dev':dev_metrics,'v2_dev_same_run':baseline_metrics,
        'dev_without_comment_overlap':metrics(y_dev[unseen],dev_pred[unseen]) if unseen.any() else None,
        'selection_caveat':'CV used for model selection; dev reused during iterative development; no new Kaggle score available',
        'submission_rows_verified':len(saved),'prediction_counts':{int(k):int(v) for k,v in saved.spam_label.value_counts().items()},
    }
    (output/'metrics_v3.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    print(f'Đã kiểm tra và lưu {len(saved)} dòng: {csv_path}',flush=True)
    return report


## 4. Chạy huấn luyện và xuất CSV

Giữ `DATA_DIR = None` để tự tìm bốn file CSV dưới `/kaggle/input`. Nếu có nhiều bản cùng tên, điền đúng thư mục chứa bộ dữ liệu muốn dùng. Trên máy cá nhân/Colab, điền đường dẫn dữ liệu. CSV để nộp sẽ ở `/kaggle/working/submission_v3.csv` trên Kaggle.


In [ ]:
DATA_DIR = None
OUTPUT_DIR = "/kaggle/working" if Path("/kaggle/working").is_dir() else "output_v3"
args = ["--output-dir", OUTPUT_DIR]
if DATA_DIR is not None:
    args += ["--data-dir", DATA_DIR]
result = main_v3(args)


In [ ]:
submission = pd.read_csv(Path(OUTPUT_DIR) / "submission_v3.csv")
display(submission.head())
print("Số dòng:", len(submission))
print("Dev v2:", result["v2_dev_same_run"]["macro_f1"])
print("Dev v3:", result["dev"]["macro_f1"])
